# Figure 1 of the paper, stable against unstable ODE

**The task in plain words.** Figure 1 puts a contracting ODE (top row, "Stable") next to a chaotic one (bottom row,
"Unstable"). Both are 128-dimensional random RNNs with the same matrix W (entries N(0, 1/128), seed 42):

- stable: dx/dt = (-x + 0.1 W tanh(x) + u(t)) / 4, with u(t) a smooth drive in a 2-dimensional subspace;
  every pair of trajectories approaches at rate at least (1 - 0.1 ||W||_2)/4 = 0.20.
- unstable: dx/dt = -x + 3.5 W tanh(x), autonomous, largest Lyapunov exponent about 0.17.

- **A, the ODE.** The solution and two neighbours from perturbed initial states, projected on their first two
  principal components, over a faint field of the projected velocities.
- **B, the merit functional** L[s] = 1/2 int_0^30 ||s' - f(s,t)||^2 dt on the plane x*(t) + alpha v1(t) + beta v2(t).
  v1 and v2 are the flattest and the steepest of the responses to a forcing pulse in t in (0, 1); their curvatures
  are 0.69 and 0.89 (stable, condition number 1.3) against 3e-6 and 16 (unstable, condition number 5.4e6). The
  height is log10(1 + 10 L / sqrt(H1 H2)), so the two rows' heights are not on one scale.
- **C, the parallel-in-time solve** of the T = 120 trajectory cut into 480 segments of h = 0.25, each integrated by
  eight DOP853 steps: s_j = phi(s_{j-1}). DEER is Newton's method on this system,
  s_j^(i+1) = phi(s_{j-1}^i) + J_j (s_{j-1}^(i+1) - s_{j-1}^i) with J_j = d phi / dx.
  The coloured curve is state coordinate 76 of the iterate, the dashed curve the sequential reference.

B and C are two problems with the same solution (T = 30 merit, T = 120 shooting system); B shows the conditioning,
C is not a descent on B's surface.

**How to use it.** The code is the package `experiments/fig1` (NumPy, Matplotlib and Pillow). Its `data/`
holds every array, so the cells below only draw; `README.txt` in that folder gives the commands that recompute the
arrays. The last cell writes the figure into `experiments/fig1/output/deer/`.

## Setup: the knobs and the package

In [ ]:
ITERATIONS = [0, 1, 5]               # the DEER iterations shown in column C, top to bottom

import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "parode").is_dir())
PKG = REPO / "experiments/fig1"
sys.path.insert(0, str(PKG))
from IPython.display import Image, display
import make_figure, simulate_all

CACHE = PKG / "data"
OUT = PKG / "output" / "deer"
simulate_all.prepare(CACHE)

## The figure

The labels are drawn by matplotlib. The layout constants are at the top of `make_figure.py`: `ROW_H` (both rows the same
height), `COLS` (the frames; column A's is square and its field fills it, with no axes), `MERIT_X` and `MERIT_Y`
(where each landscape sits in its frame), `TRAJ_*` (column C), `HEAD_Y` and `LETTER_DX` (the letters A, B, C sit on
one line, each the same distance right of its column's left edge). `STAR_PT` in `style.py` is the size of the
ODE-solution star, on the surfaces and in the legend. The surfaces are embedded as one 400 dpi image each; everything else is vector.

In [ ]:
make_figure.save_figure(CACHE, OUT, 400, ITERATIONS)
display(Image(filename=str(OUT / "figure1_horizontal_deer_text.png"), width=1000))